# Task 1 composer classification by fine-tuning MidiBERT-Piano (Colab GPU)

Fine-tunes the pretrained MidiBERT-Piano BERT-base backbone (`wazenmai/MIDI-BERT` on GitHub) on the eight-composer classification task. Writes `predictions1.json`.

The notebook is laid out so that every step that can fail does so before any GPU compute is spent. In order: install dependencies, clone the repo, mount Drive, unzip data, download pretrained checkpoint, sanity-check the tokenizer on a few sample pieces, tokenize the full train and test sets, build and verify the model with pretrained weights, then train and infer.

Heuristics that have been deliberately removed from the first pass: no discriminative learning rates, no warmup schedule, no class weighting, no augmentation, no multi-seed, no test-time augmentation. The from-scratch Transformer experiment overfit hard, and the previous post-mortem suggested class weighting was the main culprit. Match the upstream `finetune.py` defaults first, beat them later.

**Before running:**
1. Upload `student_files_updated.zip` to your Google Drive at `/content/drive/MyDrive/CSE153/student_files_updated.zip` (same place as the other notebooks).
2. Runtime menu, set GPU runtime (A100 or V100 preferred).
3. Run the cells top to bottom.

Saves `predictions1.json` to the Colab working directory and to your Drive at `/content/drive/MyDrive/CSE153/predictions1.json`.

## 1. GPU check and dependencies

In [ ]:
!nvidia-smi
# miditoolkit and chorder pinned to the versions in MidiBERT-Piano's requirements.txt
# so the tokenizer behavior matches what the pretrained weights expect.
!pip -q install miditoolkit==0.1.14 chorder==0.1.2 gdown
import sys, numpy as np, torch, transformers
print('python:', sys.version.split()[0])
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('transformers:', transformers.__version__)
print('numpy:', np.__version__)
assert torch.cuda.is_available(), 'No GPU detected; set Runtime to GPU before running.'

## 2. Clone the MidiBERT-Piano repo

We import their CP tokenizer and `MidiBert` model class directly so the tokenization and the pretrained weights stay in lockstep. NumPy 1.20 removed `np.long`, which their older code still references, so it gets monkey-patched in for compatibility.

In [ ]:
import os, sys
REPO_DIR = '/content/MIDI-BERT'
if not os.path.isdir(REPO_DIR):
    !git clone --quiet --branch CP https://github.com/wazenmai/MIDI-BERT.git {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np
if not hasattr(np, 'long'):
    np.long = int   # older MidiBERT-Piano code uses the deprecated np.long alias

from data_creation.prepare_data.model import CP
from data_creation.prepare_data import utils as cp_utils
from MidiBERT.model import MidiBert
from MidiBERT.finetune_model import SequenceClassification

CP_DICT_PATH = os.path.join(REPO_DIR, 'data_creation/prepare_data/dict/CP.pkl')
assert os.path.exists(CP_DICT_PATH), f'CP dictionary missing at {CP_DICT_PATH}'
print('repo cloned at', REPO_DIR)
print('CP dict path:', CP_DICT_PATH)
print('default pitch range used by their tokenizer:',
      'pitch tokens 22 through 107 (see make_dict.py)')

## 3. Mount Drive and unzip the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil
ZIP_PATH = '/content/drive/MyDrive/CSE153/student_files_updated.zip'
WORKDIR = '/content/work'
assert os.path.exists(ZIP_PATH), f'Zip not found at {ZIP_PATH}; upload it to Drive or change ZIP_PATH'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
if not os.path.isdir('student_files'):
    !unzip -q -o {ZIP_PATH} -d {WORKDIR}
    if os.path.isdir(os.path.join(WORKDIR, '__MACOSX')):
        shutil.rmtree(os.path.join(WORKDIR, '__MACOSX'))

DATAROOT = 'student_files/task1_composer_classification'
import json
train_meta = eval(open(os.path.join(DATAROOT, 'train.json')).read())
test_paths_meta = eval(open(os.path.join(DATAROOT, 'test.json')).read())
print('Train pieces:', len(train_meta), 'Test pieces:', len(test_paths_meta))
print('First three train entries:', list(train_meta.items())[:3])
print('First three test paths:', test_paths_meta[:3])
labels_seen = sorted({int(v) for v in train_meta.values()})
print('Train label set:', labels_seen, 'count_per_label:', {l: sum(1 for v in train_meta.values() if int(v) == l) for l in labels_seen})
N_CLASSES = max(labels_seen) + 1
print('N_CLASSES:', N_CLASSES)

## 4. Download pretrained checkpoint

The published Drive folder mostly contains finetuned checkpoints for the original paper's four downstream tasks. There is also a pretrain-only checkpoint that is what we actually want for fine-tuning on our composer set. We download the whole folder (about 350 MB), inspect what `.ckpt` files are there, and prefer one with `pretrain` in its path. If only finetuned checkpoints are available the next cell will say so loudly rather than silently picking the wrong one.

In [ ]:
CKPT_DIR = '/content/midibert_ckpts'
DRIVE_CKPT_CACHE = '/content/drive/MyDrive/CSE153/midibert_ckpts'

# Cache the downloaded checkpoint folder back to Drive so a runtime restart
# does not redo the gdown each time.
if os.path.isdir(DRIVE_CKPT_CACHE):
    print('using cached ckpt folder from Drive')
    if not os.path.isdir(CKPT_DIR):
        shutil.copytree(DRIVE_CKPT_CACHE, CKPT_DIR)
else:
    os.makedirs(CKPT_DIR, exist_ok=True)
    FOLDER_URL = 'https://drive.google.com/drive/folders/1ceIfC1UugZQHPgpEEMkdAF0VhZ1EeLl3'
    !gdown --folder {FOLDER_URL} -O {CKPT_DIR}
    os.makedirs(os.path.dirname(DRIVE_CKPT_CACHE), exist_ok=True)
    shutil.copytree(CKPT_DIR, DRIVE_CKPT_CACHE, dirs_exist_ok=True)
    print('cached to Drive at', DRIVE_CKPT_CACHE)

ckpt_files = []
for root, _, files in os.walk(CKPT_DIR):
    for fn in files:
        if fn.endswith('.ckpt'):
            full = os.path.join(root, fn)
            ckpt_files.append((full, os.path.getsize(full)))
print(f'found {len(ckpt_files)} .ckpt files:')
for path, size in sorted(ckpt_files):
    print(f'  {size//(1024*1024):4d} MB  {path}')

pretrain_candidates = [p for p, _ in ckpt_files if 'pretrain' in p.lower()]
if pretrain_candidates:
    CKPT_PATH = sorted(pretrain_candidates, key=os.path.getsize, reverse=True)[0]
    USING_PRETRAIN = True
elif ckpt_files:
    print('\nNo `pretrain` checkpoint found. Falling back to the largest finetuned checkpoint.')
    print('The backbone weights are shared, but the classification head will be re-initialized.')
    CKPT_PATH = sorted(ckpt_files, key=lambda x: x[1], reverse=True)[0][0]
    USING_PRETRAIN = False
else:
    raise RuntimeError('No .ckpt file found in the downloaded folder; cannot proceed.')

print(f'\nselected checkpoint: {CKPT_PATH}')
print(f'using as pretrain (backbone + head match)?: {USING_PRETRAIN}')

## 5. Tokenizer sanity check on a few sample pieces

Before tokenizing all 400+ MIDIs (about 10 minutes of CPU), run the CP tokenizer on a small random sample to surface any unexpected failure modes: pitches outside the pretrained vocabulary's 22 to 107 range, missing tempo events, empty MIDIs, weird time signatures. If something is going to break the bulk run, we want it to break here.

In [ ]:
import random, traceback
import miditoolkit

PITCH_LO, PITCH_HI = 22, 107   # MidiBERT-Piano's CP pitch vocabulary

cp_model = CP(dict=CP_DICT_PATH)
print('CP vocab sizes per field:', {k: len(v) for k, v in cp_model.event2word.items()})

random.seed(0)
sample_train_paths = random.sample(list(train_meta.keys()), min(8, len(train_meta)))
sample_test_paths  = random.sample(list(test_paths_meta), min(4, len(test_paths_meta)))

def tokenizer_diagnostic(rel_path):
    """Return a (status, info) tuple. Status is 'ok', 'no_notes', 'no_tempo', 'pitch_out_of_range', 'error'."""
    abs_path = os.path.join(DATAROOT, rel_path)
    try:
        m = miditoolkit.midi.parser.MidiFile(abs_path)
    except Exception as e:
        return 'error', f'parse: {e}'
    if not m.tempo_changes:
        return 'no_tempo', 'no tempo events'
    pitches = [n.pitch for inst in m.instruments for n in inst.notes]
    if not pitches:
        return 'no_notes', 'no notes'
    p_min, p_max = min(pitches), max(pitches)
    oob = sum(1 for p in pitches if p < PITCH_LO or p > PITCH_HI)
    try:
        events = cp_model.extract_events(abs_path, task='composer')
    except Exception as e:
        return 'error', f'extract_events: {e}\n{traceback.format_exc(limit=2)}'
    if events is None or len(events) == 0:
        return 'no_notes', 'extract_events returned empty'
    return 'ok', f'pitch_range=[{p_min}, {p_max}] oob_notes={oob}/{len(pitches)} events={len(events)}'

print('\n--- train sample ---')
for p in sample_train_paths:
    s, info = tokenizer_diagnostic(p)
    print(f'  [{s}] {p}: {info}')
print('\n--- test sample ---')
for p in sample_test_paths:
    s, info = tokenizer_diagnostic(p)
    print(f'  [{s}] {p}: {info}')

print('\nIf any sample is `error` or `no_tempo` and that pattern is common, '
      'the bulk tokenizer call will fail on those pieces. The bulk cell handles each piece in a try/except '
      'and reports counts, so it will not abort the whole run.')

## 6. Tokenize the full train and test sets

For each piece: extract CP events, convert to word ids, clip pitches outside the vocabulary range, slice into windows of `MAX_SEQ_LEN=512` tokens, drop or pad the final window. Track which windows came from which original piece so the val split can be honest at the piece level and so test-time inference can average over windows of the same piece.

Errors per file are caught and counted; bulk tokenization keeps going even if a handful of pieces fail.

In [ ]:
from tqdm.auto import tqdm

MAX_SEQ_LEN = 512
PAD_WORD = [cp_model.event2word[etype][f'{etype} <PAD>'] for etype in cp_model.event2word]
print('pad word per field:', PAD_WORD)

# Cache tokenized arrays to Drive so reruns are instant.
TOKEN_CACHE = '/content/drive/MyDrive/CSE153/task1_midibert_tokens.npz'

def piece_to_chunks(rel_path):
    """Tokenize one MIDI into (chunks, num_chunks) where chunks is a list of length-MAX_SEQ_LEN
    rows of 4-element token tuples. Returns (None, error_string) on any failure."""
    abs_path = os.path.join(DATAROOT, rel_path)
    try:
        events = cp_model.extract_events(abs_path, task='composer')
    except Exception as e:
        return None, f'extract_events: {e}'
    if not events:
        return None, 'no events'
    # Match the upstream prepare_data path: convert events to token ids and clip pitches in-range.
    word_rows = []
    for note_tuple in events:
        row = []
        for e in note_tuple:
            text = f'{e.name} {e.value}'
            tok = cp_model.event2word[e.name].get(text)
            if tok is None and e.name == 'Pitch':
                clipped = max(PITCH_LO, min(PITCH_HI, int(e.value)))
                text = f'Pitch {clipped}'
                tok = cp_model.event2word['Pitch'].get(text)
            if tok is None:
                return None, f'oov token: {text}'
            row.append(tok)
        word_rows.append(row)

    chunks = [word_rows[i:i+MAX_SEQ_LEN] for i in range(0, len(word_rows), MAX_SEQ_LEN)]
    # Drop the final chunk if it is shorter than half a window (matches their composer-task rule);
    # otherwise pad with the PAD word.
    if len(chunks[-1]) < MAX_SEQ_LEN:
        if len(chunks[-1]) < MAX_SEQ_LEN // 2 and len(chunks) > 1:
            chunks.pop()
        else:
            chunks[-1] = chunks[-1] + [PAD_WORD] * (MAX_SEQ_LEN - len(chunks[-1]))
    return chunks, None


def tokenize_paths(rel_paths, labels=None):
    """Tokenize a list of pieces. Returns (X, y, piece_ids, failures).
    X: (N_chunks, MAX_SEQ_LEN, 4) int32, y: (N_chunks,) int (-1 if no labels),
    piece_ids: (N_chunks,) int pointing into rel_paths, failures: list of (rel_path, reason)."""
    X_rows, y_rows, piece_idx, failures = [], [], [], []
    for i, p in enumerate(tqdm(rel_paths, desc='tokenize')):
        chunks, err = piece_to_chunks(p)
        if chunks is None:
            failures.append((p, err))
            continue
        for c in chunks:
            X_rows.append(c)
            piece_idx.append(i)
            y_rows.append(int(labels[p]) if labels is not None else -1)
    X = np.array(X_rows, dtype=np.int32)
    y = np.array(y_rows, dtype=np.int64)
    piece_ids = np.array(piece_idx, dtype=np.int32)
    return X, y, piece_ids, failures


if os.path.exists(TOKEN_CACHE):
    print('loading cached tokens from', TOKEN_CACHE)
    cache = np.load(TOKEN_CACHE, allow_pickle=True)
    X_train_all, y_train_all = cache['X_train'], cache['y_train']
    train_piece_ids = cache['train_piece_ids']
    X_test_all = cache['X_test']
    test_piece_ids = cache['test_piece_ids']
    train_piece_paths = list(cache['train_piece_paths'])
    test_piece_paths = list(cache['test_piece_paths'])
    print('cached arrays loaded.')
else:
    train_piece_paths = list(train_meta.keys())
    test_piece_paths  = list(test_paths_meta)
    print(f'tokenizing {len(train_piece_paths)} train pieces ...')
    X_train_all, y_train_all, train_piece_ids, train_fails = tokenize_paths(train_piece_paths, labels=train_meta)
    print(f'  -> {X_train_all.shape[0]} train chunks, failures: {len(train_fails)}')
    if train_fails[:5]:
        print('  first failures:', train_fails[:5])
    print(f'tokenizing {len(test_piece_paths)} test pieces ...')
    X_test_all, _, test_piece_ids, test_fails = tokenize_paths(test_piece_paths, labels=None)
    print(f'  -> {X_test_all.shape[0]} test chunks, failures: {len(test_fails)}')
    if test_fails:
        print('  test failures:', test_fails)
    np.savez(TOKEN_CACHE,
             X_train=X_train_all, y_train=y_train_all,
             train_piece_ids=train_piece_ids,
             X_test=X_test_all, test_piece_ids=test_piece_ids,
             train_piece_paths=np.array(train_piece_paths, dtype=object),
             test_piece_paths=np.array(test_piece_paths, dtype=object))
    print('cached to', TOKEN_CACHE)

print('\nFinal shapes:')
print('  X_train:', X_train_all.shape, 'dtype', X_train_all.dtype)
print('  y_train:', y_train_all.shape, 'distribution:', dict(zip(*np.unique(y_train_all, return_counts=True))))
print('  X_test :', X_test_all.shape)
print('  unique train pieces represented:', len(set(train_piece_ids.tolist())))
print('  unique test  pieces represented:', len(set(test_piece_ids.tolist())))
assert set(test_piece_ids.tolist()) == set(range(len(test_piece_paths))), \
    'some test pieces produced zero chunks; will need a fallback prediction for them'

## 7. Build the model and verify the pretrained weights load cleanly

Build the same `MidiBert` + `SequenceClassification` stack the upstream `finetune.py` uses, load the pretrained checkpoint with `strict=False`, and abort immediately if any backbone encoder weights are missing. A silent load failure here would mean training from scratch dressed up as a fine-tune.

In [ ]:
import pickle
from transformers import BertConfig

DEVICE = torch.device('cuda')
HIDDEN_SIZE = 768            # MidiBERT-Piano BERT-base default
LAYER_INDEX = -1             # last hidden layer, matches their --index_layer=12 default

with open(CP_DICT_PATH, 'rb') as f:
    e2w, w2e = pickle.load(f)
print('CP vocab sizes per field:', {k: len(v) for k, v in e2w.items()})

bert_cfg = BertConfig(
    max_position_embeddings=MAX_SEQ_LEN,
    position_embedding_type='relative_key_query',
    hidden_size=HIDDEN_SIZE,
)
midibert = MidiBert(bertConfig=bert_cfg, e2w=e2w, w2e=w2e)

ckpt = torch.load(CKPT_PATH, map_location='cpu')
state_dict = ckpt['state_dict']

# Finetune checkpoints from this repo are saved as TokenClassification / SequenceClassification,
# so backbone keys are prefixed with 'midibert.'. Pretrain checkpoints are saved directly from MidiBert,
# without that prefix. Strip 'midibert.' if present so we can load into MidiBert either way.
stripped = {}
for k, v in state_dict.items():
    if k.startswith('midibert.'):
        stripped[k[len('midibert.'):]] = v
    else:
        stripped[k] = v
# Drop head keys from finetuned checkpoints (head shape does not match ours).
stripped = {k: v for k, v in stripped.items() if not k.startswith('classifier.') and not k.startswith('attention.')}

result = midibert.load_state_dict(stripped, strict=False)
missing, unexpected = result.missing_keys, result.unexpected_keys
print(f'load result: missing={len(missing)} unexpected={len(unexpected)}')
print('first 5 missing keys :', missing[:5])
print('first 5 unexpected   :', unexpected[:5])

# The four components that actually carry MidiBERT-Piano's pretraining signal:
# the CP field embeddings (word_emb.0..3), the linear projection that merges them (in_linear),
# the BERT encoder layers, and the input word/position embeddings.
critical_prefixes = ('bert.encoder.', 'bert.embeddings.word_embeddings',
                     'word_emb.', 'in_linear.')
critical_missing = [k for k in missing if k.startswith(critical_prefixes)]
if critical_missing:
    raise RuntimeError(
        f'pretrained weights did not load for {len(critical_missing)} critical keys; '
        f'first few: {critical_missing[:5]}'
    )
print('OK: all pretraining-critical weights loaded.')

model = SequenceClassification(midibert, class_num=N_CLASSES, hs=HIDDEN_SIZE).to(DEVICE)
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nmodel: {n_total/1e6:.1f}M params total, {n_train/1e6:.1f}M trainable')

## 8. Honest piece-level split, build loaders

A piece that produces five chunks must have all five chunks in the same fold; otherwise the val number is meaningless. Verified explicitly below: zero piece-id overlap between train and val.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED = 0
BATCH_SIZE = 8           # BERT-base + seq 512 takes ~10 GB at batch 8 on A100; lower for V100/T4
LR = 2e-5                # MidiBERT-Piano finetune default
WEIGHT_DECAY = 0.01      # AdamW default in their trainer
EPOCHS = 10
EARLY_STOP_PATIENCE = 3

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

# Stratified split at the piece level: one label per piece (composer is constant per piece).
piece_labels = np.array([int(train_meta[p]) for p in train_piece_paths])
piece_idx = np.arange(len(train_piece_paths))
tr_pieces, va_pieces = train_test_split(
    piece_idx, test_size=0.10, random_state=SEED, stratify=piece_labels)
tr_pieces, va_pieces = set(tr_pieces.tolist()), set(va_pieces.tolist())
assert tr_pieces.isdisjoint(va_pieces), 'piece overlap between splits!'

tr_mask = np.array([pid in tr_pieces for pid in train_piece_ids])
va_mask = np.array([pid in va_pieces for pid in train_piece_ids])
X_tr, y_tr = X_train_all[tr_mask], y_train_all[tr_mask]
X_va, y_va = X_train_all[va_mask], y_train_all[va_mask]
pids_va = train_piece_ids[va_mask]
print(f'train chunks: {len(X_tr)} from {len(tr_pieces)} pieces')
print(f'val   chunks: {len(X_va)} from {len(va_pieces)} pieces')
print('train label distribution:', dict(zip(*np.unique(y_tr, return_counts=True))))
print('val   label distribution:', dict(zip(*np.unique(y_va, return_counts=True))))


class ChunkDataset(Dataset):
    def __init__(self, X, y=None, piece_ids=None):
        self.X = X
        self.y = y
        self.piece_ids = piece_ids
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        item = {
            'x': torch.from_numpy(np.asarray(self.X[i], dtype=np.int64)),
            'pid': int(self.piece_ids[i]) if self.piece_ids is not None else -1,
        }
        if self.y is not None:
            item['y'] = int(self.y[i])
        return item


loader_tr = DataLoader(ChunkDataset(X_tr, y_tr, train_piece_ids[tr_mask]),
                       batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
loader_va = DataLoader(ChunkDataset(X_va, y_va, pids_va),
                       batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
loader_te = DataLoader(ChunkDataset(X_test_all, None, test_piece_ids),
                       batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'\nloaders: train batches={len(loader_tr)} val={len(loader_va)} test={len(loader_te)}')

## 9. Train

Match upstream defaults: single AdamW(lr=2e-5), `CrossEntropyLoss`, no schedule, no class weighting. Per-epoch val accuracy is computed at the chunk level (matches their evaluation) and at the piece level (averages logits across all chunks of a piece before argmax, which is what we will do at test time too). The piece-level number is what predicts the leaderboard outcome.

Best-by-piece-val-accuracy state is saved both to local working dir and to Drive so a runtime disconnect mid-training does not lose the run.

In [ ]:
import torch.nn as nn

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
crit = nn.CrossEntropyLoss()

BEST_LOCAL = '/content/work/midibert_best.ckpt'
BEST_DRIVE = '/content/drive/MyDrive/CSE153/task1_midibert_best.ckpt'

def collect_logits(loader):
    model.eval()
    chunks_logits, chunks_pids, chunks_y = [], [], []
    with torch.no_grad():
        for batch in loader:
            x = batch['x'].to(DEVICE)
            attn = torch.ones((x.size(0), x.size(1)), device=DEVICE)
            logits = model(x, attn, LAYER_INDEX)
            chunks_logits.append(logits.cpu())
            chunks_pids.append(batch['pid'].numpy())
            if 'y' in batch:
                chunks_y.append(batch['y'].numpy())
    logits = torch.cat(chunks_logits).numpy()
    pids = np.concatenate(chunks_pids)
    y = np.concatenate(chunks_y) if chunks_y else None
    return logits, pids, y

def piece_accuracy(logits, pids, y, piece_paths, piece_meta):
    """Average logits over chunks of the same piece, argmax, compare to piece label."""
    correct, total = 0, 0
    for piece_id in np.unique(pids):
        mask = pids == piece_id
        avg_logit = logits[mask].mean(axis=0)
        pred = int(np.argmax(avg_logit))
        true = int(piece_meta[piece_paths[int(piece_id)]])
        correct += int(pred == true)
        total += 1
    return correct / total

best_piece_va = -1.0
best_state = None
bad_epochs = 0

for ep in range(1, EPOCHS + 1):
    model.train()
    running, nb = 0.0, 0
    pbar = tqdm(loader_tr, desc=f'ep{ep}/{EPOCHS}')
    for batch in pbar:
        x = batch['x'].to(DEVICE)
        y = batch['y'].to(DEVICE)
        attn = torch.ones((x.size(0), x.size(1)), device=DEVICE)
        opt.zero_grad()
        logits = model(x, attn, LAYER_INDEX)
        loss = crit(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        running += loss.item(); nb += 1
        pbar.set_postfix(loss=f'{running/nb:.4f}')

    va_logits, va_pids, va_y = collect_logits(loader_va)
    va_chunk_acc = float((np.argmax(va_logits, axis=1) == va_y).mean())
    va_piece_acc = piece_accuracy(va_logits, va_pids, None, train_piece_paths, train_meta)
    print(f'[ep{ep}] train_loss={running/nb:.4f}  val_chunk_acc={va_chunk_acc:.4f}  val_piece_acc={va_piece_acc:.4f}')

    if va_piece_acc > best_piece_va:
        best_piece_va = va_piece_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
        torch.save({'state_dict': best_state, 'val_piece_acc': best_piece_va, 'epoch': ep}, BEST_LOCAL)
        os.makedirs(os.path.dirname(BEST_DRIVE), exist_ok=True)
        shutil.copy(BEST_LOCAL, BEST_DRIVE)
        print(f'  -> new best piece-acc {best_piece_va:.4f}, saved')
    else:
        bad_epochs += 1
        if bad_epochs >= EARLY_STOP_PATIENCE:
            print(f'no improvement for {EARLY_STOP_PATIENCE} epochs; early-stopping at ep{ep}')
            break

print(f'\nbest val piece accuracy: {best_piece_va:.4f}')

## 10. Inference and write predictions1.json

For each test piece: average logits over its chunks, argmax, write composer integer in the format the autograder expects.

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

te_logits, te_pids, _ = collect_logits(loader_te)
print(f'test chunks: {len(te_logits)} across {len(np.unique(te_pids))} pieces')

predictions = {}
for piece_id in np.unique(te_pids):
    mask = te_pids == piece_id
    avg_logit = te_logits[mask].mean(axis=0)
    pred = int(np.argmax(avg_logit))
    path = test_piece_paths[int(piece_id)]
    predictions[path] = pred

# Any test piece that produced zero chunks (rare; would have failed an assert earlier) defaults to majority class.
majority = int(np.bincount(piece_labels).argmax())
missing_preds = [p for p in test_piece_paths if p not in predictions]
for p in missing_preds:
    predictions[p] = majority
if missing_preds:
    print(f'warning: filled {len(missing_preds)} missing test predictions with majority class {majority}')

out_local = '/content/work/predictions1.json'
with open(out_local, 'w') as f:
    f.write(repr(predictions) + '\n')
print(f'wrote {out_local}, entries={len(predictions)}')

drive_out = '/content/drive/MyDrive/CSE153/predictions1.json'
os.makedirs(os.path.dirname(drive_out), exist_ok=True)
shutil.copy(out_local, drive_out)
print(f'copied to {drive_out}')

print('\nprediction distribution:', dict(zip(*np.unique(list(predictions.values()), return_counts=True))))
print('train label distribution:', dict(zip(*np.unique(piece_labels, return_counts=True))))

In [ ]:
# sanity check on output format
d = eval(open(out_local).read())
assert len(d) == len(test_piece_paths), f'expected {len(test_piece_paths)} entries, got {len(d)}'
assert all(isinstance(v, int) and 0 <= v < N_CLASSES for v in d.values()), 'malformed predictions'
print('OK: predictions1.json has', len(d), 'integer labels in [0,', N_CLASSES, ')')

Download `/content/work/predictions1.json` from the Colab file panel, or pull it from your Drive at `/content/drive/MyDrive/CSE153/predictions1.json`. Place it in the repository root alongside the other `predictions*.json` files and submit all five artifacts to Gradescope.